# Accessing the NGEN HydroFabric on S3

**Authors:**  
   - Tony Castronova <acastronova@cuahsi.org>    
   - Irene Garousi-Nejad <igarousi@cuahsi.org>  
    
**Last Updated:** 06.19.2024   

**Description**:  

The purpose of this Jupyter Notebook is to demonstrate the process for accessing and visualizing the [NOAA Next Generation (NextGen) Water Resource Modeling Framework](https://github.com/NOAA-OWP/ngen). 

The Hydrofabric data can be accessed publicly through the AWS catalog. In this notebook, we use the **2.2** version of the dataset, which represents the most recent version available on the Amazon S3 Bucket at the time of developing this notebook (https://www.lynker-spatial.com/data/hydrofabric). The NGEN Hydrofabric is a collection of datasets used as the basis for running NGEN model simulations. It consists of: 

> - the landscape and flow network discritizations
> - the connectivity (topology) of the network features
> - the locations where information will be reported (nexus’s)
> - the divide and flowpath attributes needed to support hydrologic modeling and routing
>
> *https://noaa-owp.github.io/hydrofabric/articles/01-intro-deep-dive.html*


![Enterprise HydroFabric System](./img/enterprise-hf.png)




---

**Software Requirements**:  

The software and operating system versions used to develop this notebook are listed below. To avoid encountering issues related to version conflicts among Python packages, we recommend creating a new environment variable and installing the required packages specifically for this notebook.


> 
  fiona: 1.9.3  
  fsspec: 2023.4.0  
  geopandas: 0.12.2   
  ipyleaflet: 0.17.2  
  ipywidgets: 7.7.5   
  matplotlib: 3.7.1  
  sidecar: 0.7.0  
  shapely: 2.0.7  

---


In [ ]:
import time
import fiona
import fsspec
import shapely
import geopandas
import ipyleaflet
import geopandas as gpd
from sidecar import Sidecar
from ipywidgets import Layout
from shapely.geometry import box, shape
from pyproj import Transformer

import matplotlib.pyplot as plt
import subprocess

import warnings
warnings.filterwarnings("ignore")

import os
os.environ['AWS_NO_SIGN_REQUEST'] = 'YES'

## 1. Inspect HydroFabric Geometries

The Hydrofabric geometries can be inspected programatically because they are stored in cloud native GeoPackage GIS files. libraries. 

The data that we'll be using this notebook can be accessed directyl from the `lynker-spatial` [website](https://www.lynker-spatial.com/data). See the [NextGen Hydrofabric](https://mikejohnson51.github.io/hyAggregate/) help pages for more information regarding these data. These data are somewhat large (about 4GiB) and consists of several GIS layers:

> flowpaths  
> divides  
> lakes  
> nexus  
> pois  
> hydrolocations  
> flowpath-attributes  
> flowpath-attributes-ml  
> network   
> divide-attributes    

Define the path for accessing the NGEN HydroFabric data. The `/vsis3` prefix to leverage GDAL's Virtual File System (VSI) interface for reading files directly from S3. This allows us to interact with the HydroFabric without downloading it first.

In [ ]:
s3_gpkg_path = '/vsis3/lynker-spatial/hydrofabric/v2.2/conus/conus_nextgen.gpkg'

We can list the layers that are available in the geopackage using the GDAL `ogrinfo` utility.

In [ ]:
result = subprocess.run(
    ["ogrinfo", s3_gpkg_path],
    capture_output=True,
    text=True
)
print(result.stdout)

The data stored in the hydrofabric covers a large area (the Continental USA in our case) and as a result accessing the entire collection is slow and memory intensive. For example, the `divides` layer in the region depicted in the image below contains over 30,000 features. This is problematic for viewing within the notebook. 

![VPU 16](./img/vpu16.png)


Instead we'll use Geopandas to load portions of these data so we can inspect them. First, let's explore how we can subset these data using a bounding box. The NGEN hydrofabric geopackage file contains data in the `EPSG:5070 - NAD83 / Conus Albers` coordinate reference system. We'll need to define our bounding box using these coordinates.

To learn more about this coordinate reference system, see it's definition [here](https://spatialreference.org/ref/epsg/5070/).

In [ ]:
# define a bounding box for querying our data in EPSG:5070
bbox_of_interest = (-1309545, 2039371, -1284973, 2073668) 

Let's load the `divides` features within this bounding box using GeoPandas.

In [ ]:
divides = geopandas.read_file(s3_gpkg_path, layer='divides', bbox=bbox_of_interest)

We can quickly visualize these data using built-in GeoPandas functionality.

In [ ]:
divides.plot();

We can also look at the attributes that we're returned.

In [ ]:
divides

Now we can do the same thing for the `flowpaths` layer.

In [ ]:
reaches = geopandas.read_file(s3_gpkg_path, layer='flowpaths', bbox=bbox_of_interest)
reaches.plot()
reaches

Working with a layer's native coordinate reference system can sometimes be challenging, especially if you're not familiar with it. Here's how we can define a bounding box in another coordinate system.

In [ ]:
# define a bounding box in EPSG:4296
bbox_of_interest = (-95.146623, 45.801999, -94.46547, 46.198844)

# transform this into the coordinate system of the layer
transformer = Transformer.from_crs('EPSG:4296', 'EPSG:5070', always_xy=True)
minx, miny = transformer.transform(bbox_of_interest[0], bbox_of_interest[1])
maxx, maxy = transformer.transform(bbox_of_interest[2], bbox_of_interest[3])
bbox_in_layer_crs = (minx, miny, maxx, maxy)

# load the divides layer and reproject back into EPSG:4296
divides = geopandas.read_file(s3_gpkg_path, layer='divides', bbox=bbox_in_layer_crs)
divides = divides.to_crs(epsg='4269')

# load the reaches layer and reproject back into EPSG:4296
reaches = geopandas.read_file(s3_gpkg_path, layer='flowpaths', bbox=bbox_in_layer_crs)
reaches = reaches.to_crs(epsg='4269')

Plot the `divides` and `reaches` on the same figure.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

divides.plot(ax=ax, color='green', alpha=0.3, edgecolor='darkgreen', label='HydroFabric Divides')
reaches.plot(ax=ax, color='blue', edgecolor='blue', alpha=0.5, label='HydroFabric Reaches')

ax.legend()
plt.show()

Now that we know some basic methods for interacting with the NGEN HydroFabric, we can create an interactive map interface for us to select our area of interest. This map will also contain USGS river gauges and NHD+ reach geometries to help us select our area of interest. 

In [ ]:
class SideCarMap():
    def __init__(self,
                 basemap=ipyleaflet.basemaps.OpenStreetMap.Mapnik,
                 polygon_gdf=None,
                 reach_gdf = None,
                 plot_polygon_gdf=False,
                 plot_reach_gdf=False,
                 name='Map'):
        self.selected_id = None
        self.selected_layer = None
        self.map = None
        self.basemap = basemap
        self.polygon_gdf = polygon_gdf
        self.plot_polygon_gdf = plot_polygon_gdf
        self.reach_gdf = reach_gdf
        self.plot_reach_gdf = plot_reach_gdf
        self.name = name
        self.polygon_layer = None
        self.reach_layer = None

    def display_map(self):
        defaultLayout=Layout(width='960px', height='940px')

        
        self.map = ipyleaflet.Map(
        basemap=ipyleaflet.basemap_to_tiles(ipyleaflet.basemaps.OpenStreetMap.Mapnik, layout=defaultLayout),
            center=(45.9163, -94.8593),
            zoom=9,
            scroll_wheel_zoom=True,
            tap=False
            )
        
        # add USGS Gages
        self.map.add_layer(
            ipyleaflet.WMSLayer(
                url='http://arcgis.cuahsi.org/arcgis/services/NHD/usgs_gages/MapServer/WmsServer',
                layers='0',
                transparent=True,
                format='image/png',
                min_zoom=8,
                max_zoom=18,
                )
        )
        
        # add NHD+ Reaches
        self.map.add_layer(
            ipyleaflet.WMSLayer(
                url='https://hydro.nationalmap.gov/arcgis/services/nhd/MapServer/WMSServer',
                layers='6',
                transparent=True,
                format='image/png',
                min_zoom=8,
                max_zoom=18,
                )
        )
        
        # bind the map handler function
        self.map.on_interaction(self.handle_map_interaction)

        draw_control = ipyleaflet.DrawControl(rectangle={"shapeOptions": {"color": "#0000FF"}})
        draw_control.circle = {}
        draw_control.polygon = {}
        draw_control.polyline = {}
        draw_control.marker = {}


            
        @draw_control.on_draw
        def handle_draw(target, action, geo_json):            
            # get the bbox
            bounds = shape(geo_json['geometry']).bounds
            print(f"Bounding box: {bounds}")       

            if self.polygon_layer is not None:
                self.map.remove(self.polygon_layer)
            if self.reach_layer is not None:
                self.map.remove(self.reach_layer)
                
            # convert the bounds from EPSG:4296 (from the map) into the CRS of the Hydrofabric
            transformer = Transformer.from_crs('EPSG:4296', 'EPSG:5070', always_xy=True)
            minx, miny = transformer.transform(bounds[0], bounds[1])
            maxx, maxy = transformer.transform(bounds[2], bounds[3])
            bbox_in_layer_crs = (minx, miny, maxx, maxy)
            self.add_hydrofabric(bbox_in_layer_crs)

            draw_control.clear()
        
        
        self.map.add_control(draw_control)

        sc = Sidecar(title=self.name)
        with sc:
            display(self.map)

    def add_hydrofabric(self, bounds):

        divides = geopandas.read_file(s3_gpkg_path, layer='divides', bbox=bounds)
        self.polygon_gdf = divides.to_crs(epsg='4269')
        
        reaches = geopandas.read_file(s3_gpkg_path, layer='flowpaths', bbox=bounds)
        self.reach_gdf = reaches.to_crs(epsg='4269')
        
         # add polygon features from geopandas if they are provided
        if self.polygon_gdf is not None:
            # update the map center point
            polygon = box(*self.polygon_gdf.total_bounds)
            approx_center = (polygon.centroid.y, polygon.centroid.x)
            self.map.center = approx_center

            print('Loading Polygon GDF Features...', end='')
            st = time.time()
            geo_data = ipyleaflet.GeoData(geo_dataframe = self.polygon_gdf,
                   style={'color': 'green', 'opacity':0.75, 'weight':1.9,}
                  )
            self.map.add(geo_data)
            self.polygon_layer = self.map.layers[-1]

            print(f'{time.time() - st:0.2f} sec')

        # add reach features from geopandas if they are provided
        if self.reach_gdf is not None:
            # update the map center point
            reaches = box(*self.reach_gdf.total_bounds)
            approx_center = (reaches.centroid.y, reaches.centroid.x)
            self.map.center = approx_center

            print('Loading Reach GDF Features...', end='')
            st = time.time()
            geo_data = ipyleaflet.GeoData(geo_dataframe = self.reach_gdf,
                   style={'color': 'blue', 'opacity':0.5, 'weight':1.9,}
                  )
            self.map.add(geo_data)
            self.reach_layer = self.map.layers[-1]
            print(f'{time.time() - st:0.2f} sec')
                
    def handle_map_interaction(self, **kwargs):
    
        if kwargs.get('type') == 'click':
            lat, lon = kwargs['coordinates']
            # print(f'{lat}, {lon}')
            
            # query the reach nearest this point
            point = shapely.Point(lon, lat)

            # buffer the selected point by a small degree. This
            # is a hack for now and Buffer operations should only
            # be applied in a projected coordinate system in the future.
            pt_buf = point.buffer(0.001) 

            try:

                # query the FIM reach that intersects with the point
                mask = self.polygon_gdf.intersects(pt_buf)
                print(self.polygon_gdf.loc[mask].iloc[0] is None)
                self.selected(value=self.polygon_gdf.loc[mask].iloc[0])
                
                # remove the previously selected layers
                if self.selected_layer is not None:
                    self.map.remove(self.selected_layer)

                # highlight this layer on the map
                wlayer = ipyleaflet.WKTLayer(
                    wkt_string=self.selected().geometry.wkt,
                    style={'color': 'red', 'opacity':1, 'weight':2.,})
                self.map.add(wlayer)
                self.selected_layer = self.map.layers[-1]

                # print out the attributes
                print('-----------------')
                print('Divide Attributes')
                print('-----------------')
                for name, values in self.selected().items():
                    if name != 'geometry':
                        print(f'{name}: {values}')
                        
            except Exception: 
                print('Could not find reach for selected area')

    # getter/setter for the selected reach
    def selected(self, value=None):
        if value is None:
            if self.selected_id is None:
                print('No reach is selected.\nUse the map interface to select a reach of interest')
            else:
                return self.selected_id
        else:
            self.selected_id = value


Launch the interactive map interface in `SideCar`.

In [ ]:
m = SideCarMap(name='HydroFabric Map')
m.display_map()